In [2]:
# Transilien BI Project

## Objective
"""
Analyze train punctuality and identify influencing factors such as:
- weather
- strikes
- holidays
- temporal patterns
"""

## Pipeline
"""
Raw data → Cleaning → Enrichment → Export for SAP Analytics Cloud
"""

'\nRaw data → Cleaning → Enrichment → Export for SAP Analytics Cloud\n'

In [3]:
## Import libraries

In [4]:

"""
We import the libraries required for:
- data manipulation
- feature engineering
- holiday generation
"""
import pandas as pd
import holidays
import matplotlib.pyplot as plt
import numpy as np

In [5]:
## Load the Transilien dataset

In [6]:
"""
The original dataset contains monthly punctuality indicators for Transilien lines.
"""
df = pd.read_csv("../Data/raw/ponctualite-mensuelle-transilien.csv",sep=";")
df.head()

,Date,Service,Ligne,Nom de la ligne,Taux de ponctualité,Nombre de voyageurs à l'heure pour un voyageur en retard
0,2013-01,RER,A,RER A,83.6,5.1
1,2013-01,Transilien,R,Paris Sud Est,87.2,6.8
2,2013-03,Transilien,H,Paris Nord Ouest,92.3,12.0
3,2013-04,Transilien,N,Paris Montparnasse,90.2,9.2
4,2013-05,RER,D,RER D,87.1,6.8


In [7]:
## Data cleaning

In [8]:
"""
We standardize column names and convert data types to prepare the dataset for analysis.
"""
df.columns =(
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("'","_")
    .str.replace("é","e")

    )
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   date                                                      2009 non-null   str    
 1   service                                                   2009 non-null   str    
 2   ligne                                                     2009 non-null   str    
 3   nom_de_la_ligne                                           2009 non-null   str    
 4   taux_de_ponctualite                                       2008 non-null   float64
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64
dtypes: float64(2), str(4)
memory usage: 94.3 KB


In [9]:
"""
We convert the date column into datetime format
and remove invalid date values.
"""
df["date"]=pd.to_datetime(df["date"])
df = df.dropna(subset="date")
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      2009 non-null   datetime64[us]
 1   service                                                   2009 non-null   str           
 2   ligne                                                     2009 non-null   str           
 3   nom_de_la_ligne                                           2009 non-null   str           
 4   taux_de_ponctualite                                       2008 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64       
dtypes: datetime64[us](1), float64(2), str(3)
memory usage: 94.3 KB


In [10]:
## Feature engineering

In [11]:
"""
We create temporal variables and KPIs to improve BI analysis.
"""
df["annee"]=df["date"].dt.year
df["mois"]=df["date"].dt.month
df["nom_mois"]=df["date"].dt.month_name()
df["trimestre"]=df["date"].dt.quarter
df["taux_irregularite"] = 100 - df["taux_de_ponctualite"]

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 11 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   date                                                      2009 non-null   datetime64[us]
 1   service                                                   2009 non-null   str           
 2   ligne                                                     2009 non-null   str           
 3   nom_de_la_ligne                                           2009 non-null   str           
 4   taux_de_ponctualite                                       2008 non-null   float64       
 5   nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard  2009 non-null   float64       
 6   annee                                                     2009 non-null   int32         
 7   mois                                                 

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9


In [12]:
## Holiday enrichment

In [13]:
"""
French holidays are generated dynamically using the holidays Python package.
"""
fr_holidays = holidays.France(years=df["annee"].unique()) 
"""
We create a function that calculates the number of French holidays
for a month passed as a parameter.
"""
def count_holidays_in_month(date):
    #first day of the month
    start = date.replace(day=1)
    #last day of the month using MonthEnd offset
    end = start + pd.offsets.MonthEnd(0)
    #Generate all days in the month
    days = pd.date_range(start, end, freq="D")
    #Count how many of these days are in the list of French holidays
    return sum(day.date() in fr_holidays for day in days)




In [14]:
"""We apply the function to the date column 
to create a new variable that counts the number
 of holidays in each month."""

df["nb_jours_feries_mois"] = df["date"].apply(count_holidays_in_month)
df.head()

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries_mois
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4,1
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8,1
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7,0
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8,1
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9,4


In [15]:
## Holiday feature validation

In [16]:
# Check that the new feature was created
print(df.columns)

# Display holiday counts for several months
print(df[
    ["date", "nom_mois", "nb_jours_feries_mois"]
].head(20))

# Verify there are no missing values
print(df["nb_jours_feries_mois"].isna().sum())

# Analyze holiday count distribution
df["nb_jours_feries_mois"].value_counts()

Index(['date', 'service', 'ligne', 'nom_de_la_ligne', 'taux_de_ponctualite',
       'nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard', 'annee',
       'mois', 'nom_mois', 'trimestre', 'taux_irregularite',
       'nb_jours_feries_mois'],
      dtype='str')
         date   nom_mois  nb_jours_feries_mois
0  2013-01-01    January                     1
1  2013-01-01    January                     1
2  2013-03-01      March                     0
3  2013-04-01      April                     1
4  2013-05-01        May                     4
5  2013-05-01        May                     4
6  2013-06-01       June                     0
7  2013-06-01       June                     0
8  2013-08-01     August                     1
9  2013-09-01  September                     0
10 2013-09-01  September                     0
11 2013-10-01    October                     0
12 2013-10-01    October                     0
13 2013-11-01   November                     2
14 2013-11-01   November      

nb_jours_feries_mois
1    894
0    791
2    169
4     91
3     64
Name: count, dtype: int64

In [17]:
## Seasonal feature engineering

In [18]:
"""
We create a function to determine the season
based on the month extracted from the date.
"""
def get_season(month):
    if month in [12, 1, 2]:
        return "Hiver"
    elif month in [3, 4, 5]:
        return "Printemps"
    elif month in [6, 7, 8]:
        return "Été"
    else:
        return "Automne"


In [19]:
"""
We create a seasonal variable to improve
temporal and weather-related analysis.
"""
df["saison"] = df["mois"].apply(get_season)

In [20]:
## Seasonal feature validation

In [21]:
# Display unique month-season combinations
# to verify the season mapping
print(
    df[
        ["mois", "nom_mois", "saison"]
    ]
    .drop_duplicates()
    .sort_values("mois")
)

# Analyze season distribution
print(
    df["saison"]
    .value_counts()
)

# Verify there are no missing values
print(
    df["saison"]
    .isna()
    .sum()
)

# Display the first rows of the dataset
# to confirm the new feature
df[
    ["date", "nom_mois", "saison"]
].head(20)

    mois   nom_mois     saison
0      1    January      Hiver
78     2   February      Hiver
2      3      March  Printemps
3      4      April  Printemps
4      5        May  Printemps
6      6       June        Été
33     7       July        Été
8      8     August        Été
9      9  September    Automne
11    10    October    Automne
13    11   November    Automne
17    12   December      Hiver
saison
Automne      507
Été          506
Hiver        505
Printemps    491
Name: count, dtype: int64
0


,date,nom_mois,saison
0,2013-01-01,January,Hiver
1,2013-01-01,January,Hiver
2,2013-03-01,March,Printemps
3,2013-04-01,April,Printemps
4,2013-05-01,May,Printemps
5,2013-05-01,May,Printemps
6,2013-06-01,June,Été
7,2013-06-01,June,Été
8,2013-08-01,August,Été
9,2013-09-01,September,Automne


In [22]:
## Weather enrichment

In [23]:
"""
We load the raw weather dataset from Météo-France.
This dataset contains daily weather observations for Paris.
"""
meteo = pd.read_csv("../Data/raw/meteo_gouv_1950-2024.csv", sep=";")
print(meteo.head())

   NUM_POSTE  NOM_USUEL        LAT       LON  ALTI  AAAAMMJJ   RR  QRR  TN  \
0   75101001  INNOCENTS  48.860667  2.348333    37  19500101  0.0  1.0 NaN   
1   75101001  INNOCENTS  48.860667  2.348333    37  19500102  1.8  1.0 NaN   
2   75101001  INNOCENTS  48.860667  2.348333    37  19500103  2.0  1.0 NaN   
3   75101001  INNOCENTS  48.860667  2.348333    37  19500104  0.2  1.0 NaN   
4   75101001  INNOCENTS  48.860667  2.348333    37  19500105  1.0  1.0 NaN   

   QTN  ...  FXI3S  QFXI3S  DXI3S  QDXI3S  HXI3S  QHXI3S  DRR  QDRR  \
0  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   
1  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   
2  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   
3  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   
4  NaN  ...    NaN     NaN    NaN     NaN    NaN     NaN  NaN   NaN   

   STATUS_FXI3S  STATUS_DXI3S  
0           NaN           NaN  
1           NaN           NaN  
2       

In [24]:
## Weather dataset exploration

In [25]:
# Display available weather columns
print(meteo.columns)
#Display dataset info to check data types and missing values
meteo.info()

Index(['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'QRR',
       'TN', 'QTN', 'HTN', 'QHTN', 'TX', 'QTX', 'HTX', 'QHTX', 'TM', 'QTM',
       'TNTXM', 'QTNTXM', 'TAMPLI', 'QTAMPLI', 'TNSOL', 'QTNSOL', 'TN50',
       'QTN50', 'DG', 'QDG', 'FFM', 'QFFM', 'FF2M', 'QFF2M', 'FXY', 'QFXY',
       'DXY', 'QDXY', 'HXY', 'QHXY', 'FXI', 'QFXI', 'DXI', 'QDXI', 'HXI',
       'QHXI', 'FXI2', 'QFXI2', 'DXI2', 'QDXI2', 'HXI2', 'QHXI2', 'FXI3S',
       'QFXI3S', 'DXI3S', 'QDXI3S', 'HXI3S', 'QHXI3S', 'DRR', 'QDRR',
       'STATUS_FXI3S', 'STATUS_DXI3S'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 496112 entries, 0 to 496111
Data columns (total 60 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   NUM_POSTE     496112 non-null  int64  
 1   NOM_USUEL     496112 non-null  str    
 2   LAT           496112 non-null  float64
 3   LON           496112 non-null  float64
 4   ALTI          496112 non-null  int64  
 5   AAA

In [26]:
## Weather variable selection

In [27]:
"""
We keep only the weather variables required for the BI analysis:
- date
- rainfall
- average temperature
- average wind speed
"""
meteo = meteo[
    [
        "AAAAMMJJ",
        "RR",
        "TM",
        "FFM"
    ]
]

meteo.head()

,AAAAMMJJ,RR,TM,FFM
0,19500101,0.0,NaN,NaN
1,19500102,1.8,NaN,NaN
2,19500103,2.0,NaN,NaN
3,19500104,0.2,NaN,NaN
4,19500105,1.0,NaN,NaN


In [28]:
"""
We rename technical weather columns into clearer business-friendly names.
"""

meteo.columns = [
    "date",
    "pluie_mm",
    "temperature_moyenne",
    "vent_moyen"
]

meteo.head()

,date,pluie_mm,temperature_moyenne,vent_moyen
0,19500101,0.0,NaN,NaN
1,19500102,1.8,NaN,NaN
2,19500103,2.0,NaN,NaN
3,19500104,0.2,NaN,NaN
4,19500105,1.0,NaN,NaN


In [29]:
## Weather date conversion

In [30]:
"""
We convert the weather date column from YYYYMMDD format
to a proper datetime format.
"""

meteo["date"] = pd.to_datetime(
    meteo["date"],
    format="%Y%m%d",
    errors="coerce"
)

meteo.info()

<class 'pandas.DataFrame'>
RangeIndex: 496112 entries, 0 to 496111
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   date                 496112 non-null  datetime64[us]
 1   pluie_mm             481784 non-null  float64       
 2   temperature_moyenne  62506 non-null   float64       
 3   vent_moyen           43957 non-null   float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 15.1 MB


In [31]:
## Weather temporal filtering

In [32]:
"""
We keep weather observations starting from 2013
to match the Transilien dataset period.
"""

meteo = meteo[
    (meteo["date"] >= "2013-01-01") 
    
]

meteo.head()


,date,pluie_mm,temperature_moyenne,vent_moyen
53297,2013-01-01,0.8,NaN,NaN
53298,2013-01-02,1.3,NaN,NaN
53299,2013-01-03,0.2,NaN,NaN
53300,2013-01-04,0.0,NaN,NaN
53301,2013-01-05,0.0,NaN,NaN


In [33]:
## Weather date coverage validation

In [34]:
# Verify weather date coverage
print(meteo["date"].min())
print(meteo["date"].max())

# Analyze yearly weather observations
print(
    meteo["date"]
    .dt.year
    .value_counts()
    .sort_index()
)

2013-01-01 00:00:00
2024-12-31 00:00:00
date
2013    5382
2014    5147
2015    4076
2016    3669
2017    3984
2018    3987
2019    3886
2020    3875
2021    3904
2022    3375
2023    2252
2024    2196
Name: count, dtype: int64


In [35]:
## Weather preprocessing refinement

In [36]:
"""
An initial preprocessing pipeline was created for the Météo-France dataset.

Further exploration revealed that the dataset contained observations
from multiple weather stations with inconsistent temporal coverage.

Some stations only covered a limited time range,
while others contained observations extending until 2024.
This initially led to misleading preprocessing results
and large amounts of missing values after filtering and aggregation.

The weather preprocessing pipeline was therefore refined
to identify and select a single consistent weather station
covering the full analysis period (2013–2024).
"""

'\nAn initial preprocessing pipeline was created for the Météo-France dataset.\n\nFurther exploration revealed that the dataset contained observations\nfrom multiple weather stations with inconsistent temporal coverage.\n\nSome stations only covered a limited time range,\nwhile others contained observations extending until 2024.\nThis initially led to misleading preprocessing results\nand large amounts of missing values after filtering and aggregation.\n\nThe weather preprocessing pipeline was therefore refined\nto identify and select a single consistent weather station\ncovering the full analysis period (2013–2024).\n'

In [37]:
## Weather enrichment

In [38]:
"""
We load the raw Météo-France weather dataset.
"""

meteo = pd.read_csv(
    "../Data/raw/meteo_gouv_1950-2024.csv",
    sep=";"
)

meteo.head()

,NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMMJJ,RR,QRR,TN,QTN,...,FXI3S,QFXI3S,DXI3S,QDXI3S,HXI3S,QHXI3S,DRR,QDRR,STATUS_FXI3S,STATUS_DXI3S
0,75101001,INNOCENTS,48.860667,2.348333,37,19500101,0.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,75101001,INNOCENTS,48.860667,2.348333,37,19500102,1.8,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,75101001,INNOCENTS,48.860667,2.348333,37,19500103,2.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,75101001,INNOCENTS,48.860667,2.348333,37,19500104,0.2,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,75101001,INNOCENTS,48.860667,2.348333,37,19500105,1.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
## Weather variable selection

In [40]:
"""
We keep the weather variables required for the analysis,
including station information for validation purposes.
"""

meteo = meteo[
    [
        "NUM_POSTE",
        "NOM_USUEL",
        "AAAAMMJJ",
        "RR",
        "TM",
        "FFM"
    ]
]

meteo.head()

,NUM_POSTE,NOM_USUEL,AAAAMMJJ,RR,TM,FFM
0,75101001,INNOCENTS,19500101,0.0,NaN,NaN
1,75101001,INNOCENTS,19500102,1.8,NaN,NaN
2,75101001,INNOCENTS,19500103,2.0,NaN,NaN
3,75101001,INNOCENTS,19500104,0.2,NaN,NaN
4,75101001,INNOCENTS,19500105,1.0,NaN,NaN


In [41]:
## Weather date conversion

In [42]:
"""
We convert the weather date column into datetime format.
"""

meteo["AAAAMMJJ"] = pd.to_datetime(
    meteo["AAAAMMJJ"],
    format="%Y%m%d",
    errors="coerce"
)

meteo.info()

<class 'pandas.DataFrame'>
RangeIndex: 496112 entries, 0 to 496111
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   NUM_POSTE  496112 non-null  int64         
 1   NOM_USUEL  496112 non-null  str           
 2   AAAAMMJJ   496112 non-null  datetime64[us]
 3   RR         481784 non-null  float64       
 4   TM         62506 non-null   float64       
 5   FFM        43957 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(1)
memory usage: 22.7 MB


In [43]:
## Weather temporal filtering

In [44]:
"""
We keep weather observations starting from 2013
to match the Transilien dataset period.
"""

meteo = meteo[
    meteo["AAAAMMJJ"] >= "2013-01-01"
]

meteo.head()

,NUM_POSTE,NOM_USUEL,AAAAMMJJ,RR,TM,FFM
53297,75106001,LUXEMBOURG,2013-01-01,0.8,NaN,NaN
53298,75106001,LUXEMBOURG,2013-01-02,1.3,NaN,NaN
53299,75106001,LUXEMBOURG,2013-01-03,0.2,NaN,NaN
53300,75106001,LUXEMBOURG,2013-01-04,0.0,NaN,NaN
53301,75106001,LUXEMBOURG,2013-01-05,0.0,NaN,NaN


In [45]:
"""
We rename technical weather columns into clearer business-friendly names.
"""

meteo.columns = [
    "num_poste",
    "nom_usuel",
    "date",
    "pluie_mm",
    "temperature_moyenne",
    "vent_moyen"
]

meteo.head()

,num_poste,nom_usuel,date,pluie_mm,temperature_moyenne,vent_moyen
53297,75106001,LUXEMBOURG,2013-01-01,0.8,NaN,NaN
53298,75106001,LUXEMBOURG,2013-01-02,1.3,NaN,NaN
53299,75106001,LUXEMBOURG,2013-01-03,0.2,NaN,NaN
53300,75106001,LUXEMBOURG,2013-01-04,0.0,NaN,NaN
53301,75106001,LUXEMBOURG,2013-01-05,0.0,NaN,NaN


In [46]:
## Weather date coverage validation

In [47]:
# Verify weather date coverage
print(meteo["date"].min())
print(meteo["date"].max())

# Analyze yearly weather observations
print(
    meteo["date"]
    .dt.year
    .value_counts()
    .sort_index()
)

2013-01-01 00:00:00
2024-12-31 00:00:00
date
2013    5382
2014    5147
2015    4076
2016    3669
2017    3984
2018    3987
2019    3886
2020    3875
2021    3904
2022    3375
2023    2252
2024    2196
Name: count, dtype: int64


In [48]:
"""
Temperature availability varies significantly between weather stations.

Some stations provide observations until 2024,
while others contain large amounts of missing temperature data.
"""

'\nTemperature availability varies significantly between weather stations.\n\nSome stations provide observations until 2024,\nwhile others contain large amounts of missing temperature data.\n'

In [49]:
## Weather station coverage analysis

In [50]:
station_coverage = (
    meteo
    .groupby("nom_usuel")["date"]
    .agg(["min", "max", "count"])
    .sort_values("max", ascending=False)
)

station_coverage

,min,max,count
nom_usuel,,,
LONGCHAMP,2013-01-01,2024-12-31,4366
LUXEMBOURG,2013-01-01,2024-12-31,4263
TOUR EIFFEL,2013-01-01,2024-12-31,4217
PARIS-MONTSOURIS-DOUBLE,2016-08-25,2024-12-31,3051
LARIBOISIERE,2013-01-01,2024-12-31,4353
PARIS-MONTSOURIS,2013-01-01,2024-12-31,4383
BUTTES CHAUMONT,2013-01-01,2023-01-31,3622
ST-ANTOINE,2013-01-01,2023-01-31,3406
SALPETRIERE,2013-01-01,2022-12-31,3621


In [51]:
## Final weather station selection

In [52]:
"""
We select the PARIS-MONTSOURIS weather station
because it provides consistent weather observations
across the full analysis period (2013–2024).
"""

meteo = meteo[
    meteo["nom_usuel"] == "PARIS-MONTSOURIS"
]

print(meteo.head())
meteo.info()

        num_poste         nom_usuel       date  pluie_mm  temperature_moyenne  \
271362   75114001  PARIS-MONTSOURIS 2013-01-01       1.8                  7.6   
271363   75114001  PARIS-MONTSOURIS 2013-01-02       0.8                  6.6   
271364   75114001  PARIS-MONTSOURIS 2013-01-03       0.2                  9.7   
271365   75114001  PARIS-MONTSOURIS 2013-01-04       0.0                  9.8   
271366   75114001  PARIS-MONTSOURIS 2013-01-05       0.0                  9.0   

        vent_moyen  
271362         3.1  
271363         2.4  
271364         2.5  
271365         2.2  
271366         2.2  
<class 'pandas.DataFrame'>
Index: 4383 entries, 271362 to 275744
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   num_poste            4383 non-null   int64         
 1   nom_usuel            4383 non-null   str           
 2   date                 4383 non-null   datetime64[us]
 3   

In [53]:
## Weather station cleanup

In [54]:
"""
We remove weather station metadata
after selecting the final reference station.
"""

meteo = meteo.drop(
    columns=[
        "num_poste",
        "nom_usuel"
    ]
)

meteo.head()

,date,pluie_mm,temperature_moyenne,vent_moyen
271362,2013-01-01,1.8,7.6,3.1
271363,2013-01-02,0.8,6.6,2.4
271364,2013-01-03,0.2,9.7,2.5
271365,2013-01-04,0.0,9.8,2.2
271366,2013-01-05,0.0,9.0,2.2


In [55]:
## Weather feature validation

In [56]:
"""
We verify the completeness of the selected weather station
before continuing the preprocessing pipeline.
"""

meteo[["pluie_mm", "temperature_moyenne", "vent_moyen"]].isna().sum()

pluie_mm                0
temperature_moyenne     0
vent_moyen             15
dtype: int64

In [57]:
"""
The selected weather station shows a high level of data completeness,
with only a small number of missing wind observations remaining.
"""

'\nThe selected weather station shows a high level of data completeness,\nwith only a small number of missing wind observations remaining.\n'

In [58]:
## Monthly weather aggregation

In [59]:
"""
We create monthly time keys to aggregate daily weather observations
at the same granularity as the Transilien dataset.
"""

meteo["annee"] = meteo["date"].dt.year
meteo["mois"] = meteo["date"].dt.month

meteo.head()

,date,pluie_mm,temperature_moyenne,vent_moyen,annee,mois
271362,2013-01-01,1.8,7.6,3.1,2013,1
271363,2013-01-02,0.8,6.6,2.4,2013,1
271364,2013-01-03,0.2,9.7,2.5,2013,1
271365,2013-01-04,0.0,9.8,2.2,2013,1
271366,2013-01-05,0.0,9.0,2.2,2013,1


In [60]:
"""
We create monthly time keys to aggregate daily weather observations
at the same granularity as the Transilien dataset.
"""

meteo["annee"] = meteo["date"].dt.year
meteo["mois"] = meteo["date"].dt.month

meteo.head()

,date,pluie_mm,temperature_moyenne,vent_moyen,annee,mois
271362,2013-01-01,1.8,7.6,3.1,2013,1
271363,2013-01-02,0.8,6.6,2.4,2013,1
271364,2013-01-03,0.2,9.7,2.5,2013,1
271365,2013-01-04,0.0,9.8,2.2,2013,1
271366,2013-01-05,0.0,9.0,2.2,2013,1


In [61]:
"""
We aggregate daily weather observations at monthly level.

Temperature and wind are averaged over the month,
while rainfall is summed to represent total monthly precipitation.
"""

meteo_mensuelle = (
    meteo
    .groupby(["annee", "mois"], as_index=False)
    .agg({
        "temperature_moyenne": "mean",
        "pluie_mm": "sum",
        "vent_moyen": "mean"
    })
)

meteo_mensuelle.head(20)

,annee,mois,temperature_moyenne,pluie_mm,vent_moyen
0,2013,1,4.174194,42.2,2.800000
1,2013,2,3.321429,37.4,3.492857
2,2013,3,5.545161,36.6,3.106452
3,2013,4,11.040000,29.7,3.363333
4,2013,5,12.587097,111.2,2.861290
5,2013,6,17.370000,64.0,3.026667
6,2013,7,22.719355,47.8,2.993548
7,2013,8,20.554839,37.6,2.438710
8,2013,9,17.213333,49.5,2.280000
9,2013,10,14.245161,43.4,2.806452


In [62]:
## Weather and Transilien merge

In [63]:

df.head()

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries_mois,saison
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4,1,Hiver
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8,1,Hiver
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7,0,Printemps
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8,1,Printemps
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9,4,Printemps


In [64]:
"""
We merge monthly weather indicators
with the Transilien dataset.
"""

df = df.merge(
    meteo_mensuelle,
    on=["annee", "mois"],
    how="left"
)

df.head()

,date,service,ligne,nom_de_la_ligne,taux_de_ponctualite,nombre_de_voyageurs_à_l_heure_pour_un_voyageur_en_retard,annee,mois,nom_mois,trimestre,taux_irregularite,nb_jours_feries_mois,saison,temperature_moyenne,pluie_mm,vent_moyen
0,2013-01-01,RER,A,RER A,83.6,5.1,2013,1,January,1,16.4,1,Hiver,4.174194,42.2,2.800000
1,2013-01-01,Transilien,R,Paris Sud Est,87.2,6.8,2013,1,January,1,12.8,1,Hiver,4.174194,42.2,2.800000
2,2013-03-01,Transilien,H,Paris Nord Ouest,92.3,12.0,2013,3,March,1,7.7,0,Printemps,5.545161,36.6,3.106452
3,2013-04-01,Transilien,N,Paris Montparnasse,90.2,9.2,2013,4,April,2,9.8,1,Printemps,11.040000,29.7,3.363333
4,2013-05-01,RER,D,RER D,87.1,6.8,2013,5,May,2,12.9,4,Printemps,12.587097,111.2,2.861290


In [65]:
## Weather feature validation

In [66]:
""" Check missing values after weather merge""" 
df[
    [
        "temperature_moyenne",
        "pluie_mm",
        "vent_moyen"
    ]
].isna().sum()

temperature_moyenne    195
pluie_mm               195
vent_moyen             195
dtype: int64

In [67]:
""" Check missing values after weather merge by year"""
df.groupby("annee")[
    [
        "temperature_moyenne",
        "pluie_mm",
        "vent_moyen"
    ]
].apply(lambda x: x.isna().sum())

,temperature_moyenne,pluie_mm,vent_moyen
annee,,,
2013,0,0,0
2014,0,0,0
2015,0,0,0
2016,0,0,0
2017,0,0,0
2018,0,0,0
2019,0,0,0
2020,0,0,0
2021,0,0,0


In [68]:
"""
The weather merge was successfully validated.

Missing values are primarily associated with observations
outside the temporal coverage of the weather dataset,
confirming the consistency of the merge process.
"""

'\nThe weather merge was successfully validated.\n\nMissing values are primarily associated with observations\noutside the temporal coverage of the weather dataset,\nconfirming the consistency of the merge process.\n'

In [72]:
"""
We remove observations from 2025 and 2026
to ensure temporal consistency
with the weather dataset coverage.
"""
df = df[
    (df["annee"] != 2025)
    &
    (df["annee"] != 2026)
]
print((df["annee"] >= 2025).sum())

0
